# Qwen2.5-Coder-0.5B (BASE, decoder-only) — 2-GPU DDP, Kaggle

Same dataset, same rewritten instructions, same system prompt, same evaluation harness as the
CodeT5+ 220M runs — but a genuinely different architecture family: decoder-only, autoregressive,
not T5's encoder-decoder span-corruption pretraining. This is the cleanest test yet of whether the
generate/HumanEval plateau is about the T5 family specifically, or something more fundamental about
how the task is framed.

## Why this notebook's training mechanics are genuinely different, not just a model-name swap
A decoder-only model has **no separate encoder and decoder** — prompt and target share ONE sequence.
This changes real things, not cosmetic ones:
- **Training format:** prompt + `<OUTPUT>` + target are concatenated into a single sequence; the loss
  is masked (`-100`) over the prompt portion so only the target contributes to the gradient.
- **Truncation direction:** if the combined sequence is too long, the **prompt** is trimmed (keeping
  its end, closest to `<OUTPUT>`) — never the target, since losing the actual code to generate would
  be far worse than losing some leading prompt context. **Unit-tested** against a hand-built fake
  tokenizer across a full sweep of lengths, including the edge case where the target alone exceeds
  the budget, before being trusted in this notebook.
- **A new boundary token, `<OUTPUT>`**, marks where the answer starts — CodeT5+ didn't need this
  because the encoder/decoder split already provided that boundary structurally.
- **Generation must slice off the echoed prompt:** a decoder-only model's output includes everything
  you fed it, so only the newly generated tokens (`seq[:, input_len:]`) are kept.
- **System prompt is plain prepended text, not a chat template** — this is the BASE (non-Instruct)
  model, so `apply_chat_template` doesn't apply; fine-tuning is what teaches the association here,
  exactly as it did for CodeT5+.
- **Mean-initialization of new special tokens handles tied vs. untied embeddings generically** —
  checks `model.config.tie_word_embeddings` at runtime rather than assuming either case.

## Before running (on kaggle.com)
1. `dataset_formats_v2.parquet` attached via **"Add Input"**.
2. Settings -> Accelerator -> **GPU T4 x2**.
3. Settings -> Internet -> **On**.

## Training (true 2-GPU DDP, same proven mechanism as the CodeT5+ notebooks)
`Salesforce` -- no, `Qwen/Qwen2.5-Coder-0.5B` (base), format `tgt_b_canonical`, lr 1e-5 (the proven
full-fine-tune learning rate for this exact model from Phase 1 — not guessed), per-GPU batch 4x4
(lower batch than CodeT5+'s 8x2 since this model is 2.2x the params and holds the full concatenated
sequence in memory at once), 7 max epochs with early stopping, `notebook_launcher` for real
distributed training across both GPUs (not the slower automatic `DataParallel` fallback).

## Evaluation
Identical, previously-proven harness: subprocess-isolated HumanEval (64) and ClassEval (10), plus
held-out own-dataset exact-match/validity checks. Greedy pass@1 is the primary, honest metric;
compile-validated generation (beam + retry-until-valid) is reported separately and explicitly labeled
as NOT a correctness measure — the same honest distinction maintained throughout this project.

## Important
**Not executed end to end** here (real 2-GPU training with live model downloads isn't something this
delivery environment can run). The label-masking and truncation logic — the part most likely to have
a subtle bug in a from-scratch causal-LM training setup — WAS unit-tested directly, including the
found-and-fixed edge case where a target alone exceeds the token budget. Run it and read Cell 10's
comparison against every prior run.

In [ ]:
# CELL 0 — INSTALL (pinned for consistency with the rest of this project's environment)
!pip install -q "transformers==4.46.3" "accelerate>=0.26.0" datasets pandas pyarrow
print('Installed. RESTART THE SESSION NOW (Run -> Restart Session), then run Cell 1 onward.')

In [ ]:
# CELL 1 — IMPORTS + CONFIG
# CUDA-safety note (same as the CodeT5+ dual-GPU notebooks): nothing here may touch CUDA before
# Cell 6's notebook_launcher runs.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # must be set BEFORE torch import

import ast, copy, gc, hashlib, json, math, random, re, signal, subprocess, sys, tempfile, time, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore", category=SyntaxWarning)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

PARQUET_FILENAME = "dataset_formats_v2.parquet"
KAGGLE_INPUT_DIR = Path("/kaggle/input")
WORK = Path("/kaggle/working/qwen05b_base")
WORK.mkdir(parents=True, exist_ok=True)
BEST_MODEL_DIR = WORK / "best_model"
RESULTS_FILE = WORK / "results.json"

# BASE model, not -Instruct -- this is a decoder-only causal LM, a genuinely different
# architecture family from CodeT5+'s encoder-decoder. The same model id Phase 1 already
# validated loads and trains correctly in this environment.
BASE_MODEL = "Qwen/Qwen2.5-Coder-0.5B"
TARGET_FORMAT_COL = "tgt_b_canonical"
FITS_COL = "fits512_b"

# <OUTPUT> is new here: a causal LM reads prompt and target as ONE sequence, so it needs an
# explicit boundary token marking "the answer starts here" -- CodeT5+ didn't need this because
# the encoder/decoder split already provided that boundary structurally.
NEW_SPECIAL_TOKENS = ["<GENERATE>", "<REPAIR>", "<REFACTOR>", "<CODE>", "</CODE>", "<ERROR>", "</ERROR>", "<OUTPUT>"]

# Same system prompt as the CodeT5+ run, same reasoning: this is the BASE (non-chat) model, so it
# is prepended as plain text, not passed through a chat template -- apply_chat_template is for the
# -Instruct variant, which was trained to treat system/user roles specially. A base model has no
# such training; fine-tuning on this fixed text block is what teaches the association here.
SYSTEM_PROMPT = (
    "You are a Python code generation assistant for software engineering tasks spanning API design, "
    "database access, data structures and algorithms, and data science code. Rules: output only "
    "complete, syntactically valid Python with every import included. Never include explanations or "
    "text outside the code. Fully implement every function, method, and class -- never stop early or "
    "leave a placeholder. Where relevant, follow SOLID principles, RESTful API conventions, and "
    "proper error handling."
)

EPOCHS = 7
LR = 1e-5                 # the proven full-fine-tune lr for THIS exact model from Phase 1 (gave
                          # 29.7% HumanEval there; reused rather than guessing a new value)
BATCH_SIZE = 4            # PER GPU -- lower than CodeT5+'s 8: Qwen 0.5B is 2.2x the params, and
                          # causal-LM training holds the full concatenated prompt+target sequence
                          # in memory at once (no separate encoder/decoder split)
GRAD_ACCUM = 4            # keeps the same effective batch as before despite the smaller per-step batch
MAX_SEQ_LEN = 768         # prompt + <OUTPUT> + target combined, as ONE sequence -- measured for
                          # real in Cell 3, not assumed

HE_N = 64
CLASSEVAL_N = 10
CLASSEVAL_MAX_PROMPT_TOKENS = 450
EVAL_OWN_N = 300
GEN_BATCH = 16
RETRY_TEMPERATURES = [0.2, 0.5, 0.8]

N_GPUS = torch.cuda.device_count()
print(f"torch {torch.__version__} | GPUs visible: {N_GPUS}")
if N_GPUS == 0:
    raise RuntimeError(
        "No GPU found. In the right sidebar: Settings -> Accelerator -> GPU T4 x2, and Internet -> On. "
        "The session restarts; then run this cell again."
    )
print(f"Base model: {BASE_MODEL} (decoder-only) | format: {TARGET_FORMAT_COL} | epochs: {EPOCHS}")

## Load the dataset

In [ ]:
# CELL 2 — FIND + LOAD dataset_formats_v2.parquet
candidates = list(KAGGLE_INPUT_DIR.rglob(PARQUET_FILENAME))
if not candidates:
    print(f"Could not find {PARQUET_FILENAME} under {KAGGLE_INPUT_DIR}. What's attached:")
    if KAGGLE_INPUT_DIR.exists():
        for p in KAGGLE_INPUT_DIR.rglob("*"):
            if p.is_file():
                print(f"  {p}")
    raise FileNotFoundError("Attach the dataset with this file via 'Add Input', then re-run this cell.")
src = candidates[0]
print(f"Using: {src} ({src.stat().st_size/1e6:.1f} MB)")

df = pd.read_parquet(src)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

need = ["id", "unit_id", "split", "task", "instruction", "context_code", "error", TARGET_FORMAT_COL, FITS_COL, "ok_b"]
missing = [c for c in need if c not in df.columns]
assert not missing, f"expected columns missing: {missing}"

mask = df[FITS_COL].astype(bool) & df["ok_b"].astype(bool)
before = len(df)
df = df[mask].reset_index(drop=True)
print(f"Kept rows with a valid, within-budget {TARGET_FORMAT_COL}: {before} -> {len(df)}")
print("tasks:", dict(df.task.value_counts()))

## Build the prompt text, measure the real combined-sequence token cost

In [ ]:
# CELL 3 — BUILD THE PROMPT TEXT, MEASURE REAL COMBINED-SEQUENCE TOKEN COST (Qwen's own tokenizer)
def build_prompt_text(row):
    task = row["task"]
    if task == "generate":
        body = f"<GENERATE> {row['instruction']}"
    elif task == "repair":
        body = (f"<REPAIR> {row['instruction']}\n<CODE>\n{row.get('context_code','')}\n</CODE>"
                f"\n<ERROR>\n{row.get('error','')}\n</ERROR>")
    elif task == "refactor":
        body = f"<REFACTOR> {row['instruction']}\n<CODE>\n{row.get('context_code','')}\n</CODE>"
    else:
        raise ValueError(f"unknown task: {task}")
    return f"{SYSTEM_PROMPT}\n\n{body}\n<OUTPUT>\n"

df["prompt_text"] = df.apply(build_prompt_text, axis=1)

from transformers import AutoTokenizer
_tok_probe = AutoTokenizer.from_pretrained(BASE_MODEL)
if _tok_probe.pad_token is None:
    _tok_probe.pad_token = _tok_probe.eos_token

sample = df.sample(min(1000, len(df)), random_state=SEED)
combined_lens = [len(_tok_probe(p + t)["input_ids"])
                 for p, t in zip(sample["prompt_text"], sample[TARGET_FORMAT_COL])]
fits_pct = sum(1 for l in combined_lens if l <= MAX_SEQ_LEN) / len(combined_lens) * 100
print(f"Combined (prompt + target) sequence length with Qwen's tokenizer:")
print(f"  mean {np.mean(combined_lens):.0f} | median {np.median(combined_lens):.0f} | "
      f"p90 {np.percentile(combined_lens, 90):.0f} | max {max(combined_lens)}")
print(f"  fits {MAX_SEQ_LEN} tokens: {fits_pct:.1f}% (measured on a {len(sample)}-row sample)")

by_task_fit = {}
for task in ("generate", "repair", "refactor"):
    sub = df[df.task == task].sample(min(300, (df.task == task).sum()), random_state=SEED)
    lens = [len(_tok_probe(p + t)["input_ids"]) for p, t in zip(sub["prompt_text"], sub[TARGET_FORMAT_COL])]
    by_task_fit[task] = sum(1 for l in lens if l <= MAX_SEQ_LEN) / len(lens) * 100
    print(f"  {task:9s} fits {MAX_SEQ_LEN}: {by_task_fit[task]:.1f}%  (mean {np.mean(lens):.0f} tokens)")
if min(by_task_fit.values()) < 85:
    print(f"\nWARNING: some task's combined sequences commonly exceed {MAX_SEQ_LEN} tokens -- raise")
    print("MAX_SEQ_LEN in Cell 1 if GPU memory allows, since truncation here would cut off the END")
    print("of the target (the actual code), which is worse than cutting off part of the prompt.")
del _tok_probe

## Define the training function (runs once per GPU -- not called yet)

In [ ]:
# CELL 4 — DEFINE THE TRAINING FUNCTION (runs once per GPU, launched by Cell 6's notebook_launcher)
def build_labeled_sequence(tokenizer, prompt_text, target_text, max_len):
    """Concatenate prompt + target into ONE sequence (causal LM has no separate encoder/decoder),
    mask the prompt portion with -100 so loss is computed ONLY on the target. If the combined
    sequence is too long, truncate the PROMPT from the left (keep the end of it, which is closest
    to <OUTPUT> and most informative) rather than the target -- cutting off the actual code the
    model is supposed to produce would be far worse than losing some leading prompt context.
    In the pathological case where the target ALONE meets or exceeds max_len (not expected at
    MAX_SEQ_LEN=768 given this dataset's typical target lengths, but handled explicitly rather
    than left to an unchecked edge case), there is no room for any prompt at all -- the target's
    own end (which includes eos) is kept intact and only its START is trimmed if truly necessary."""
    target_ids = tokenizer(target_text + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
    if len(target_ids) >= max_len:
        kept = target_ids[-max_len:]
        return kept, kept
    room_for_prompt = max_len - len(target_ids)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    if len(prompt_ids) > room_for_prompt:
        prompt_ids = prompt_ids[-room_for_prompt:]
    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids
    return input_ids, labels

def train_qwen05b():
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, EarlyStoppingCallback
    from datasets import Dataset
    from accelerate import PartialState

    state = PartialState()
    is_main = state.is_main_process
    if is_main:
        print(f"[rank {state.process_index}] starting on {state.num_processes} process(es)")

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    vocab_before = len(tokenizer)
    tokenizer.add_special_tokens({"additional_special_tokens": NEW_SPECIAL_TOKENS})
    if is_main:
        print(f"Tokenizer vocab: {vocab_before} -> {len(tokenizer)}")

    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
    tied = model.config.tie_word_embeddings
    old_in = model.get_input_embeddings().weight.data.clone()
    mean_in = old_in.mean(dim=0)
    old_out_emb = model.get_output_embeddings()
    mean_out = old_out_emb.weight.data.clone().mean(dim=0) if (old_out_emb is not None and not tied) else None

    model.resize_token_embeddings(len(tokenizer))
    with torch.no_grad():
        in_emb = model.get_input_embeddings()
        for t in NEW_SPECIAL_TOKENS:
            tid = tokenizer.convert_tokens_to_ids(t)
            in_emb.weight.data[tid] = mean_in.clone()
        if not tied:
            out_emb = model.get_output_embeddings()
            if out_emb is not None:
                for t in NEW_SPECIAL_TOKENS:
                    tid = tokenizer.convert_tokens_to_ids(t)
                    out_emb.weight.data[tid] = mean_out.clone()
    if is_main:
        print(f"New special tokens mean-initialized. tie_word_embeddings={tied} "
              f"({'single embedding updated' if tied else 'input AND output embeddings updated separately'})")
        total_p = sum(p.numel() for p in model.parameters())
        print(f"Total params: {total_p/1e6:.1f}M")

    def tokenize_split(frame, desc):
        all_ids, all_labels = [], []
        for prompt, target in zip(frame["prompt_text"], frame[TARGET_FORMAT_COL]):
            ids, labels = build_labeled_sequence(tokenizer, prompt, target, MAX_SEQ_LEN)
            all_ids.append(ids)
            all_labels.append(labels)
        ds = Dataset.from_dict({"input_ids": all_ids, "labels": all_labels})
        if is_main:
            print(f"{desc}: {len(ds)} examples tokenized")
        return ds

    train_df_local = df[df.split == "train"].reset_index(drop=True)
    val_df_local = df[df.split == "val"].reset_index(drop=True)
    train_ds = tokenize_split(train_df_local, "train")
    val_ds = tokenize_split(val_df_local, "val")

    def collate(batch):
        max_len = max(len(b["input_ids"]) for b in batch)
        input_ids, attention_mask, labels = [], [], []
        for b in batch:
            pad_len = max_len - len(b["input_ids"])
            input_ids.append(b["input_ids"] + [tokenizer.pad_token_id] * pad_len)
            attention_mask.append([1] * len(b["input_ids"]) + [0] * pad_len)
            labels.append(b["labels"] + [-100] * pad_len)
        return {"input_ids": torch.tensor(input_ids), "attention_mask": torch.tensor(attention_mask),
                "labels": torch.tensor(labels)}

    args = TrainingArguments(
        output_dir=str(WORK / "checkpoints"),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        warmup_ratio=0.1,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        fp16=True,
        gradient_checkpointing=True,
        logging_steps=20,
        report_to="none",
        seed=SEED,
        ddp_find_unused_parameters=False,
    )

    trainer = Trainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=collate, processing_class=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    if is_main:
        print(f"Training: {EPOCHS} max epochs, lr={LR}, per-GPU batch={BATCH_SIZE}x{GRAD_ACCUM}, "
              f"{state.num_processes} GPU(s) via real DDP")
    train_result = trainer.train()

    if is_main:
        print(f"\nTraining complete. {train_result}")
        trainer.save_model(str(BEST_MODEL_DIR))
        tokenizer.save_pretrained(str(BEST_MODEL_DIR))
        print(f"Saved best checkpoint to {BEST_MODEL_DIR}")
        history = [{k: v for k, v in log.items() if k in ("epoch", "loss", "eval_loss")}
                   for log in trainer.state.log_history if "eval_loss" in log or "loss" in log]
        (WORK / "train_history.json").write_text(json.dumps(history, indent=1))
        print(f"Saved training history ({len(history)} log entries)")

print("Training function defined (not yet run -- Cell 6 launches it on all visible GPUs).")

## Launch true 2-GPU DDP training

In [ ]:
# CELL 5 — LAUNCH TRUE DDP TRAINING ACROSS ALL VISIBLE GPUs
from accelerate import notebook_launcher

t0 = time.time()
notebook_launcher(train_qwen05b, num_processes=N_GPUS)
print(f"\nTotal training wall-clock time: {(time.time()-t0)/60:.1f} minutes")

assert BEST_MODEL_DIR.exists(), "training did not produce a saved model -- check the output above for errors"
print(f"Best model confirmed on disk at {BEST_MODEL_DIR}")

## Reload the trained model for evaluation (single GPU, main process)

In [ ]:
# CELL 6 — RELOAD THE TRAINED MODEL FOR EVALUATION (single GPU, main process)
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda:0"
tokenizer = AutoTokenizer.from_pretrained(str(BEST_MODEL_DIR))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # required for correct batched generation with a decoder-only model
                                  # (training used right-padding internally via the custom collator;
                                  # this is purely a tokenizer setting, safe to change independently)

model = AutoModelForCausalLM.from_pretrained(str(BEST_MODEL_DIR)).to(DEVICE)
model.eval()
print(f"Reloaded trained model from {BEST_MODEL_DIR} onto {DEVICE}")
for t in NEW_SPECIAL_TOKENS:
    tid = tokenizer.convert_tokens_to_ids(t)
    assert tid != tokenizer.unk_token_id, f"{t} did not survive the save/reload as a real token"
print("Special tokens confirmed present after reload.")

## Evaluation kit (subprocess-isolated, hard timeouts, proven against reference solutions)

In [ ]:
# CELL 7 — EVALUATION KIT (subprocess-isolated, hard timeouts, proven against reference solutions)
PRELUDE = ("import resource\n"
           "try:\n    resource.setrlimit(resource.RLIMIT_AS, (3 << 30, 3 << 30))\nexcept Exception:\n    pass\n")

def run_python(files, entry, args=(), timeout=10):
    with tempfile.TemporaryDirectory() as d:
        for name, text in files.items():
            Path(d, name).write_text((PRELUDE + text) if name == entry else text, encoding="utf-8")
        proc = subprocess.Popen([sys.executable, entry, *map(str, args)], cwd=d, stdin=subprocess.DEVNULL,
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE, encoding="utf-8",
                                errors="replace", start_new_session=True)
        try:
            out, err = proc.communicate(timeout=timeout)
            return proc.returncode, out, err
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except Exception:
                pass
            try:
                proc.communicate(timeout=5)
            except Exception:
                pass
            return -9, "", "timeout"

HE_RUNNER = '''
import json, sys
spec = json.load(open("spec.json"))
src = open("prog.py", encoding="utf-8").read()
ns = {"__name__": "heval_module"}
try:
    exec(compile(src, "prog.py", "exec"), ns)
    ns["check"](ns[spec["entry"]])
except BaseException as e:
    sys.stderr.write(type(e).__name__ + ": " + str(e)[:300] + "\\n")
    sys.exit(1)
print("__PASS__")
'''

CE_RUNNER = '''
import io, json, sys, unittest
src = open("gen.py", encoding="utf-8").read() + "\\n\\n" + open("test.py", encoding="utf-8").read()
ns = {"__name__": "classeval_module"}
try:
    exec(compile(src, "prog.py", "exec"), ns)
except BaseException as e:
    print(json.dumps({"status": "exec_fail", "error": (type(e).__name__ + ": " + str(e))[:200]})); sys.exit(0)
classes = [v for v in ns.values() if isinstance(v, type) and issubclass(v, unittest.TestCase) and v is not unittest.TestCase]
total = failed = 0
for cls in classes:
    res = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(unittest.defaultTestLoader.loadTestsFromTestCase(cls))
    total += res.testsRun
    failed += len(res.failures) + len(res.errors)
print(json.dumps({"status": "ok", "tests": total, "failed": failed}))
'''

def top_level_imports(src):
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return "\n".join(l for l in src.splitlines() if l.startswith(("import ", "from ")))
    keep = [n for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom))]
    return ast.unparse(ast.Module(body=keep, type_ignores=[])) if keep else ""

def humaneval_header(prompt, entry):
    try:
        tree = ast.parse(prompt)
        keep = [n for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom))
                or (isinstance(n, ast.FunctionDef) and n.name != entry)]
        return ast.unparse(ast.Module(body=keep, type_ignores=[])) if keep else ""
    except SyntaxError:
        return top_level_imports(prompt)

def run_humaneval(prompt, entry, test, gen_src, timeout=20):
    prog = humaneval_header(prompt, entry) + "\n\n" + gen_src + "\n\n" + test + "\n"
    rc, out, err = run_python({"runner.py": HE_RUNNER, "prog.py": prog, "spec.json": json.dumps({"entry": entry})},
                              "runner.py", timeout=timeout)
    return rc == 0 and "__PASS__" in out

def run_classeval(skeleton, test, gen_src, timeout=30):
    gen = top_level_imports(skeleton) + "\n\n" + gen_src
    rc, out, err = run_python({"runner.py": CE_RUNNER, "gen.py": gen, "test.py": test}, "runner.py", timeout=timeout)
    if rc == -9:
        return {"status": "timeout", "tests": 0, "failed": 0}
    try:
        return json.loads(out.strip().splitlines()[-1])
    except Exception:
        return {"status": "runner_error", "tests": 0, "failed": 0}

def is_valid_python(code_str):
    try:
        tree = ast.parse(code_str)
    except Exception:
        return False
    try:
        compile(tree, "<e>", "exec")
        return True
    except Exception:
        return False

def defines(src, name, kind):
    try:
        return any(isinstance(n, kind) and n.name == name for n in ast.walk(ast.parse(src)))
    except Exception:
        return False

def canonical_humaneval_src(row):
    tree = ast.parse(row["prompt"] + row["canonical_solution"])
    node = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == row["entry_point"])
    return ast.unparse(node)

def load_humaneval_rows(source):
    if source == "hf":
        from datasets import load_dataset
        return [dict(r) for r in load_dataset("evalplus/humanevalplus", split="test")]
    import gzip, urllib.request
    raw = urllib.request.urlopen("https://raw.githubusercontent.com/openai/human-eval/master/data/HumanEval.jsonl.gz", timeout=60).read()
    return [json.loads(l) for l in gzip.decompress(raw).decode("utf-8").splitlines() if l.strip()]

def load_classeval_rows(source):
    if source == "hf":
        from datasets import load_dataset
        return [dict(r) for r in load_dataset("FudanSELab/ClassEval", split="test")]
    import urllib.request
    return json.loads(urllib.request.urlopen("https://raw.githubusercontent.com/FudanSELab/ClassEval/master/data/ClassEval_data.json", timeout=60).read().decode("utf-8"))

def select_humaneval(n, seed):
    best = None
    for source in ("hf", "github"):
        try:
            rows = load_humaneval_rows(source)
        except Exception as e:
            print(f"  HumanEval from {source}: unavailable ({type(e).__name__}: {str(e)[:80]})")
            continue
        idx = list(range(len(rows)))
        random.Random(seed).shuffle(idx)
        step = len(idx) / n
        chosen = [rows[i] for i in sorted(idx[int(k * step)] for k in range(n))]
        good = []
        for r in chosen:
            try:
                ok = run_humaneval(r["prompt"], r["entry_point"], r["test"], canonical_humaneval_src(r), timeout=20)
            except Exception as e:
                print(f"    reference-check error on {r.get('task_id')}: {type(e).__name__}: {str(e)[:100]}")
                ok = False
            if ok:
                good.append(r)
        print(f"  HumanEval from {source}: {len(good)}/{len(chosen)} reference solutions pass the harness")
        if best is None or len(good) > len(best[1]):
            best = (source, good, len(chosen))
        if len(good) >= 0.9 * len(chosen):
            break
    return best

def select_classeval(n, seed, tok):
    for source in ("hf", "github"):
        try:
            rows = load_classeval_rows(source)
            break
        except Exception as e:
            print(f"  ClassEval from {source}: unavailable ({type(e).__name__}: {str(e)[:80]})")
    else:
        return None
    fits = [r for r in rows if len(tok(f"<GENERATE> Complete this Python class:\n{r['skeleton']}")["input_ids"]) <= CLASSEVAL_MAX_PROMPT_TOKENS]
    random.Random(seed).shuffle(fits)
    chosen, tried = [], 0
    for r in fits:
        if len(chosen) >= n or tried >= 60:
            break
        tried += 1
        try:
            res = run_classeval(r["skeleton"], r["test"], r["solution_code"], timeout=30)
        except Exception as e:
            print(f"    reference-check error on {r.get('task_id')}: {type(e).__name__}: {str(e)[:100]}")
            res = {"status": "error"}
        if res.get("status") == "ok" and res.get("tests", 0) > 0 and res.get("failed", 1) == 0:
            chosen.append(r)
    print(f"  ClassEval from {source}: {len(rows)} problems, {len(fits)} fit the encoder window, "
          f"tried {tried}, kept {len(chosen)} whose reference solution passes the harness")
    return source, chosen

print("Evaluation kit ready (isolated processes, hard time limits, tests must actually run).")

## Generation: greedy pass@1 (primary) + compile-validated (separate, not pass@1)

In [ ]:
# CELL 8 — GENERATION (decoder-only: the model echoes the prompt back, so output must be SLICED to
# keep only the newly generated tokens -- unlike CodeT5+, where the decoder's output never included
# the encoder's input). Same two-mechanism split as the CodeT5+ notebook: greedy pass@1 is the
# primary, honest number; compile-validated is a separate, clearly non-pass@1 measurement.

@torch.no_grad()
def generate_texts_greedy(mdl, prompts, max_new_tokens=400, batch_size=GEN_BATCH):
    mdl.eval()
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    outs = [None] * len(prompts)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        enc = tokenizer([prompts[i] for i in idx], return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
        input_len = enc["input_ids"].shape[1]
        with torch.amp.autocast("cuda", enabled=True):
            seq = mdl.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                               max_new_tokens=max_new_tokens, num_beams=1, do_sample=False,
                               pad_token_id=tokenizer.pad_token_id)
        for j, i in enumerate(idx):
            new_tokens = seq[j][input_len:]   # drop the echoed prompt -- keep only what was generated
            outs[i] = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return outs

@torch.no_grad()
def generate_compile_validated(mdl, prompt, max_new_tokens=400, num_beams=4):
    enc = tokenizer([prompt], return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
    input_len = enc["input_ids"].shape[1]
    with torch.amp.autocast("cuda", enabled=True):
        seq = mdl.generate(**enc, max_new_tokens=max_new_tokens, num_beams=num_beams, early_stopping=True,
                           pad_token_id=tokenizer.pad_token_id)
    out = tokenizer.decode(seq[0][input_len:], skip_special_tokens=True)
    if is_valid_python(out):
        return out, "beam"
    for temp in RETRY_TEMPERATURES:
        with torch.amp.autocast("cuda", enabled=True):
            seq2 = mdl.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=temp,
                                top_p=0.95, pad_token_id=tokenizer.pad_token_id)
        out2 = tokenizer.decode(seq2[0][input_len:], skip_special_tokens=True)
        if is_valid_python(out2):
            return out2, f"sample_t{temp}"
    return out, "beam_invalid"

print("Generation functions ready: generate_texts_greedy (pass@1) and generate_compile_validated (NOT pass@1).")

## Run the full evaluation

In [ ]:
# CELL 9 — RUN THE FULL EVALUATION
summary = {}

print("Selecting benchmark problems (reference solutions run through the harness first)...")
he_sel = None
try:
    he_sel = select_humaneval(HE_N, SEED)
except Exception as e:
    print(f"  HumanEval selection failed ({type(e).__name__}: {str(e)[:150]})")
he_problems = he_sel[1] if he_sel else []

ce_sel = None
try:
    ce_sel = select_classeval(CLASSEVAL_N, SEED, tokenizer)
except Exception as e:
    print(f"  ClassEval selection failed ({type(e).__name__}: {str(e)[:150]})")
ce_problems = ce_sel[1] if ce_sel else []
print(f"\nUsable: HumanEval {len(he_problems)}/{HE_N}  ClassEval {len(ce_problems)}/{CLASSEVAL_N}")

# ---------- own-dataset: greedy pass@1 (primary) ----------
random.seed(SEED)
test_df = df[df.split == "test"].reset_index(drop=True)
eval_rows = test_df.sample(min(EVAL_OWN_N, len(test_df)), random_state=SEED).reset_index(drop=True)
print(f"\nGenerating {len(eval_rows)} own-dataset outputs (greedy, pass@1)...")
t0 = time.time()
gens = generate_texts_greedy(model, eval_rows["prompt_text"].tolist())
gen_sec = time.time() - t0
print(f"  done in {gen_sec:.0f}s")

def norm_dump(code_str):
    tree = ast.parse(code_str)
    for n in ast.walk(tree):
        if isinstance(n, ast.Constant):
            n.kind = None
        if hasattr(n, "type_comment"):
            n.type_comment = None
    return ast.dump(tree)

def exact_match(pred_src, ref_src):
    try:
        return norm_dump(pred_src) == norm_dump(ref_src)
    except Exception:
        return False

own_records = []
for (_, row), text in zip(eval_rows.iterrows(), gens):
    valid = is_valid_python(text)
    exact = exact_match(text, row[TARGET_FORMAT_COL]) if valid else False
    own_records.append({"id": row["id"], "task": row["task"], "valid": valid, "exact": exact})
own_df = pd.DataFrame(own_records)

summary["own_greedy"] = {
    "n": len(own_df), "valid": int(own_df.valid.sum()), "exact": int(own_df.exact.sum()), "gen_seconds": gen_sec,
    "by_task": {t: {"n": len(g), "valid": int(g.valid.sum()), "exact": int(g.exact.sum())}
                for t, g in own_df.groupby("task")},
}
print(f"Own-dataset greedy: valid {summary['own_greedy']['valid']}/{summary['own_greedy']['n']}  "
      f"exact {summary['own_greedy']['exact']}/{summary['own_greedy']['n']}")
for t, s in summary["own_greedy"]["by_task"].items():
    print(f"  {t:9s} valid {s['valid']}/{s['n']}  exact {s['exact']}/{s['n']}")

# ---------- HumanEval: greedy pass@1 ----------
def he_prompt(p):
    return f"{SYSTEM_PROMPT}\n\n<GENERATE> Complete this Python function:\n{p}\n<OUTPUT>\n"

he_passed = 0
if he_problems:
    gens_he = generate_texts_greedy(model, [he_prompt(r["prompt"]) for r in he_problems])
    for r, text in zip(he_problems, gens_he):
        code = text if (is_valid_python(text) and defines(text, r["entry_point"], ast.FunctionDef)) else r["prompt"] + text
        has_def = is_valid_python(code) and defines(code, r["entry_point"], ast.FunctionDef)
        passed = run_humaneval(r["prompt"], r["entry_point"], r["test"], code) if has_def else False
        he_passed += bool(passed)
    print(f"\nHumanEval pass@1 (greedy): {he_passed}/{len(he_problems)} ({he_passed/len(he_problems)*100:.1f}%)")
else:
    print("\nHumanEval: skipped (no usable problems)")
summary["humaneval"] = {"n": len(he_problems), "passed": he_passed}

# ---------- ClassEval: greedy pass@1 ----------
def ce_prompt(skel):
    return f"{SYSTEM_PROMPT}\n\n<GENERATE> Complete this Python class:\n{skel}\n<OUTPUT>\n"

ce_passed = 0
if ce_problems:
    gens_ce = generate_texts_greedy(model, [ce_prompt(r["skeleton"]) for r in ce_problems])
    for r, text in zip(ce_problems, gens_ce):
        has_cls = is_valid_python(text) and defines(text, r["class_name"], ast.ClassDef)
        res = run_classeval(r["skeleton"], r["test"], text) if has_cls else {"status": "no_class", "tests": 0, "failed": 0}
        passed = res.get("status") == "ok" and res.get("tests", 0) > 0 and res.get("failed", 1) == 0
        ce_passed += bool(passed)
    print(f"ClassEval class-level pass@1 (greedy): {ce_passed}/{len(ce_problems)} ({ce_passed/len(ce_problems)*100:.1f}%)")
else:
    print("ClassEval: skipped (no usable problems)")
summary["classeval"] = {"n": len(ce_problems), "passed": ce_passed}

# ---------- compile-validated (separate, NOT pass@1) ----------
print("\n" + "=" * 70)
print("COMPILE-VALIDATED GENERATION (beam + retry-until-valid -- NOT a pass@1/correctness metric)")
print("=" * 70)
he_cv_valid = 0
if he_problems:
    for r in he_problems:
        text, method = generate_compile_validated(model, he_prompt(r["prompt"]))
        he_cv_valid += is_valid_python(text)
    print(f"HumanEval prompts: valid Python under compile-validated generation: {he_cv_valid}/{len(he_problems)} "
          f"({he_cv_valid/len(he_problems)*100:.1f}%)")
summary["humaneval_compile_validated_valid_pct"] = (he_cv_valid / len(he_problems) * 100) if he_problems else None

ce_cv_valid = 0
if ce_problems:
    for r in ce_problems:
        text, method = generate_compile_validated(model, ce_prompt(r["skeleton"]))
        ce_cv_valid += is_valid_python(text)
    print(f"ClassEval prompts: valid Python under compile-validated generation: {ce_cv_valid}/{len(ce_problems)} "
          f"({ce_cv_valid/len(ce_problems)*100:.1f}%)")
summary["classeval_compile_validated_valid_pct"] = (ce_cv_valid / len(ce_problems) * 100) if ce_problems else None

RESULTS_FILE.write_text(json.dumps(summary, indent=2, default=str))
print(f"\nSaved: {RESULTS_FILE}")

## Report — compare against every prior run

In [ ]:
# CELL 10 — REPORT: HOW DOES A DIFFERENT ARCHITECTURE FAMILY COMPARE?
history = json.loads((WORK / "train_history.json").read_text())
summary = json.loads(RESULTS_FILE.read_text())

print("=" * 90)
print("Qwen2.5-Coder-0.5B (base, decoder-only) + system prompt (DDP, 2 GPUs)")
print("=" * 90)

val_losses = [h["eval_loss"] for h in history if "eval_loss" in h]
print(f"\nValidation loss by epoch: {' -> '.join(f'{v:.3f}' for v in val_losses)}")

og = summary["own_greedy"]
print(f"\nOwn-dataset, GREEDY pass@1 (n={og['n']}):")
print(f"  valid Python: {og['valid']}/{og['n']} ({og['valid']/og['n']*100:.1f}%)")
print(f"  exact match:  {og['exact']}/{og['n']} ({og['exact']/og['n']*100:.1f}%)")
for t, s in og["by_task"].items():
    print(f"    {t:9s} valid {s['valid']}/{s['n']} ({s['valid']/s['n']*100:.1f}%)   "
          f"exact {s['exact']}/{s['n']} ({s['exact']/s['n']*100:.1f}%)")

he, ce = summary["humaneval"], summary["classeval"]
print(f"\nHumanEval pass@1 (greedy): {he['passed']}/{he['n']}" + (f" ({he['passed']/he['n']*100:.1f}%)" if he["n"] else " (skipped)"))
print(f"ClassEval pass@1 (greedy): {ce['passed']}/{ce['n']}" + (f" ({ce['passed']/ce['n']*100:.1f}%)" if ce["n"] else " (skipped)"))

hv = summary.get("humaneval_compile_validated_valid_pct")
cv = summary.get("classeval_compile_validated_valid_pct")
print(f"\nCompile-validated valid-Python rate (NOT pass@1):")
print(f"  HumanEval prompts: {hv:.1f}%" if hv is not None else "  HumanEval: skipped")
print(f"  ClassEval prompts: {cv:.1f}%" if cv is not None else "  ClassEval: skipped")

print("\n" + "=" * 90)
print("COMPARISON ACROSS EVERYTHING TESTED SO FAR (same dataset, same instructions, same format)")
print("=" * 90)
gen = og["by_task"].get("generate", {"n": 1, "valid": 0, "exact": 0})
rep = og["by_task"].get("repair", {"n": 1, "valid": 0, "exact": 0})
print(f"{'metric':<22} {'CodeT5-small':>13} {'CodeT5+220M':>13} {'+sys prompt':>13} {'Qwen0.5B base':>14}")
print(f"{'generate valid %':<22} {'68.3':>13} {'90.8':>13} {'84.3':>13} {gen['valid']/gen['n']*100:>14.1f}")
print(f"{'generate exact %':<22} {'0.0':>13} {'0.0':>13} {'0.8':>13} {gen['exact']/gen['n']*100:>14.1f}")
print(f"{'repair exact %':<22} {'16.8':>13} {'76.8':>13} {'76.0':>13} {rep['exact']/rep['n']*100:>14.1f}")
print(f"{'HumanEval %':<22} {'0.0':>13} {'0.0':>13} {'0.0':>13} {(he['passed']/he['n']*100 if he['n'] else 0):>14.1f}")
print(f"{'ClassEval %':<22} {'0.0':>13} {'0.0':>13} {'0.0':>13} {(ce['passed']/ce['n']*100 if ce['n'] else 0):>14.1f}")

print("\n" + "=" * 90)
print("VERDICT")
print("=" * 90)
he_pct = (he["passed"] / he["n"] * 100) if he["n"] else 0
gen_exact_pct = gen["exact"] / gen["n"] * 100
if gen_exact_pct > 1 or he_pct > 1:
    print(f"A genuinely different architecture family (decoder-only, autoregressive-pretrained) moved")
    print(f"generate off zero where T5-family size, framing, and validity fixes all failed to")
    print(f"(generate exact {gen_exact_pct:.1f}%, HumanEval {he_pct:.1f}%). This points to the PRETRAINING")
    print(f"OBJECTIVE, not just parameter count, being the real factor behind the earlier plateau.")
    print(f"Worth continuing to scale and tune on this architecture family specifically.")
else:
    print(f"Even a different architecture family (decoder-only, autoregressive) still shows 0% on")
    print(f"generate's exact match and HumanEval. Combined with the T5-family results (size, system")
    print(f"prompt, and compile-validation all failing to move these same two numbers), this is now")
    print(f"FOUR different levers across TWO architecture families with no movement on generate")
    print(f"correctness specifically, while repair stayed strong throughout. This points toward the")
    print(f"task itself (one-sentence instruction -> full implementation) being underdetermined, not")
    print(f"any specific model choice. Worth revisiting whether 'generate' instructions need concrete")
    print(f"behavioral specs (argument names/types, example input/output) rather than a prose description.")

## Finish (download from Kaggle's Output tab)

In [ ]:
# CELL 11 — FINISH (Kaggle: download from the Output tab)
for f in (RESULTS_FILE, WORK / "train_history.json"):
    print(f"{f}  ({f.stat().st_size/1e3:.0f} KB)")
print(f"Model checkpoint: {BEST_MODEL_DIR}  (saved, large -- see note below)")
print("\nTo get these files onto your computer:")
print("  1. Click 'Save Version' (top right) -> 'Save & Run All' -> Save.")
print("  2. Once it finishes, open this notebook's 'Output' tab.")
print("  3. results.json and train_history.json will be listed there with a Download button.")
print("  Note: the saved model checkpoint directory is large; download only results.json and")
print("  train_history.json unless you specifically need the model weights.")